# Dogecoin Forecasting Machine Learning Model


## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from joblib import dump, load
from dotenv import load_dotenv
import os

## Fetching Dogecoin data

In [ ]:
from binance.client import Client
from datetime import datetime

# Loading environmental variables from .env file
load_dotenv()

# Access to env vars
api_key = os.getenv('API_KEY')
api_secret = os.getenv('API_SECRET')

client = Client(api_key, api_secret)

def fetch_dogecoin_data():
    klines = client.get_historical_klines("DOGEEUR", Client.KLINE_INTERVAL_1DAY, "1 Jan, 2017")

    # Dataframe creation
    df = pd.DataFrame(klines, columns=[
        "timestamp", "open", "high", "low", "close", "volume", "close_time", "quote_asset_volume",
        "number_of_trades", "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ])

    df["date"] = pd.to_datetime(df["timestamp"], unit='ms')
    df["price"] = df["close"].astype(float)
    df["volume"] = df["volume"].astype(float)

    df = df[["date", "price", "volume"]]

    df.to_csv("dogecoin_data.csv", index=False)
    print("Data saved in 'dogecoin_data.csv'.")

if __name__ == "__main__":
    fetch_dogecoin_data()

## Preprocessing

In [ ]:
def prepare_data(file_path, window_size=30):
    df = pd.read_csv(file_path)
    scaler = MinMaxScaler()
    df["price"] = scaler.fit_transform(df["price"].values.reshape(-1, 1))

    sequences = []
    targets = []
    for i in range(len(df) - window_size):
        seq = df["price"].values[i:i + window_size]
        target = df["price"].values[i + window_size]
        sequences.append(seq)
        targets.append(target)

    X = np.array(sequences)
    y = np.array(targets)

    return X, y, scaler

## LSTM model Definition

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h_0 = torch.zeros(1, x.size(0), 50).to(x.device)
        c_0 = torch.zeros(1, x.size(0), 50).to(x.device)
        out, _ = self.lstm(x, (h_0, c_0))
        out = self.fc(out[:, -1, :])
        return out

## Creating the pipeline

In [ ]:
def create_pipeline():
    steps = [
        ('scaler', MinMaxScaler()),
        ('model', LSTMModel(input_size=1, hidden_size=50, output_size=1))
    ]
    pipeline = Pipeline(steps)
    return pipeline

## Training LSTM model

In [ ]:
def train_lstm_model(X, y, epochs=20, batch_size=32):
    model = LSTMModel(input_size=1, hidden_size=50, output_size=1)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(inputs.unsqueeze(-1))
            loss = criterion(outputs, targets.unsqueeze(-1))
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item()}")

    return model

## Saving model and transformer

In [ ]:
if __name__ == "__main__":
    X, y, scaler = prepare_data("dogecoin_data.csv")
    model = train_lstm_model(X, y)
    torch.save(model.state_dict(), "dogecoin_lstm_model.pth")
    dump(scaler, 'scaler.joblib')
    print("Model and scaler saved.")

## Inference

In [ ]:
def predict(data):
    model = LSTMModel(input_size=1, hidden_size=50, output_size=1)
    model.load_state_dict(torch.load("dogecoin_lstm_model.pth"))
    model.eval()
    scaler = load('scaler.joblib')

    df = pd.DataFrame(data)
    df["price"] = scaler.transform(df["price"].values.reshape(-1, 1))
    X = torch.tensor(df["price"].values, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
    with torch.no_grad():
        prediction = model(X).item()
    prediction = scaler.inverse_transform([[prediction]])
    return prediction[0][0]